# Question-Guided Router Cached-Feature Debug


In [ ]:
GITHUB_REPO_URL = "https://github.com/<your-user>/<your-repo>.git"
GITHUB_BRANCH = "huy"
USE_SAFE_CONFIG = False

import os
os.environ["GITHUB_REPO_URL"] = GITHUB_REPO_URL
os.environ["GITHUB_BRANCH"] = GITHUB_BRANCH
os.environ["USE_SAFE_CONFIG"] = "1" if USE_SAFE_CONFIG else "0"
print({"repo": GITHUB_REPO_URL, "branch": GITHUB_BRANCH, "safe_config": USE_SAFE_CONFIG})


In [ ]:
%%bash
set -e
cd /kaggle/working
if [ ! -d Efficient_VLM_For_Autonomous_Driving ]; then git clone --branch "$GITHUB_BRANCH" "$GITHUB_REPO_URL" Efficient_VLM_For_Autonomous_Driving; fi
cd Efficient_VLM_For_Autonomous_Driving
git checkout "$GITHUB_BRANCH"
git pull --ff-only
pip install -e .


In [ ]:
import os

if not os.environ.get("HF_TOKEN"):
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as exc:
        print("HF_TOKEN secret not loaded:", type(exc).__name__)

print("HF_TOKEN present:", bool(os.environ.get("HF_TOKEN")))


In [ ]:
%%bash
nvidia-smi || true
python - <<'PY'
import torch
print({'cuda': torch.cuda.is_available(), 'gpu_count': torch.cuda.device_count()})
PY


In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
if [ "$USE_SAFE_CONFIG" = "1" ]; then CONFIG=configs/repvit_t5_efficient_mini_kaggle_2gpu_router_vsa_nonorm_k49_safe.yaml; else CONFIG=configs/repvit_t5_efficient_mini_kaggle_2gpu_router_vsa_nonorm_k49.yaml; fi
python -m efficient_vlm_ad prepare-data --config "$CONFIG" --subset smoke --debug --debug-samples 2
python -m efficient_vlm_ad prepare-features --config "$CONFIG" --subset smoke --debug --debug-samples 2


In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
if [ "$USE_SAFE_CONFIG" = "1" ]; then CONFIG=configs/repvit_t5_efficient_mini_kaggle_2gpu_router_vsa_nonorm_k49_safe.yaml; else CONFIG=configs/repvit_t5_efficient_mini_kaggle_2gpu_router_vsa_nonorm_k49.yaml; fi
python -m efficient_vlm_ad debug-sample --config "$CONFIG" --split train --index 0 --debug --debug-samples 1
python -m efficient_vlm_ad diagnose-train --config "$CONFIG" --stage align --debug --debug-samples 1 --debug-numerics


In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
if [ "$USE_SAFE_CONFIG" = "1" ]; then CONFIG=configs/repvit_t5_efficient_mini_kaggle_2gpu_router_vsa_nonorm_k49_safe.yaml; PROFILE=outputs/repvit_t5_efficient_mini_kaggle_2gpu_router_vsa_nonorm_k49_safe; else CONFIG=configs/repvit_t5_efficient_mini_kaggle_2gpu_router_vsa_nonorm_k49.yaml; PROFILE=outputs/repvit_t5_efficient_mini_kaggle_2gpu_router_vsa_nonorm_k49; fi
PYTHONUNBUFFERED=1 accelerate launch --multi_gpu --num_processes 2 --num_machines 1 --mixed_precision no --dynamo_backend no -m efficient_vlm_ad train --config "$CONFIG" --stage align --max-steps 20 --debug --debug-samples 1 --debug-numerics --progress-log-every-steps 1
PYTHONUNBUFFERED=1 accelerate launch --multi_gpu --num_processes 2 --num_machines 1 --mixed_precision no --dynamo_backend no -m efficient_vlm_ad train --config "$CONFIG" --stage finetune --resume "$PROFILE/checkpoints/align_best.pt" --max-steps 20 --debug --debug-samples 1 --debug-numerics --progress-log-every-steps 1


In [ ]:
%%bash
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
if [ "$USE_SAFE_CONFIG" = "1" ]; then PROFILE=outputs/repvit_t5_efficient_mini_kaggle_2gpu_router_vsa_nonorm_k49_safe; else PROFILE=outputs/repvit_t5_efficient_mini_kaggle_2gpu_router_vsa_nonorm_k49; fi
ls -lh "$PROFILE/checkpoints" || true
tail -n 20 "$PROFILE/debug/train_progress.jsonl" || true
nvidia-smi || true
